In [12]:
from datetime import datetime, timedelta

class Flashcard:
    def __init__(self, en, eu, example_en=None, example_eu=None):
        self.en = en
        self.eu = eu
        self.example_en = example_en
        self.example_eu = example_eu
        self.interval = 1          
        self.ease_factor = 2.5       
        self.repetitions = 0
        self.last_reviewed = None
        self.next_review = datetime.now()

    def review(self, difficulty: str):
        """Update flashcard based on how difficult the recall was."""
        self.repetitions += 1
        if difficulty == "easy":
            self.ease_factor += 0.1
            self.interval *= self.ease_factor
        elif difficulty == "medium":
            self.interval *= 1.0
        else: 
            self.ease_factor = max(1.3, self.ease_factor - 0.2)
            self.interval = 1

        self.last_reviewed = datetime.now()
        self.next_review = self.last_reviewed + timedelta(days=self.interval)

    def __repr__(self):
        return f"<Flashcard {self.english} → {self.basque}>"


In [13]:
import random

class Deck:
    def __init__(self, name, cards=None):
        self.name = name
        self.cards = cards if cards else []

    def add_card(self, card: Flashcard):
        self.cards.append(card)

    def get_due_cards(self):
        """Return cards that are due for review today."""
        now = datetime.now()
        return [card for card in self.cards if card.next_review <= now]

    def random_card(self):
        """Return a random card from the deck."""
        return random.choice(self.cards)

    def __len__(self):
        return len(self.cards)


In [14]:
class ReviewSession:
    def __init__(self, deck: Deck):
        self.deck = deck
        self.queue = deck.get_due_cards()
        self.reviewed = []

    def has_next(self):
        return len(self.queue) > 0

    def next_card(self):
        if self.has_next():
            return self.queue.pop(0)
        return None

    def review_card(self, card: Flashcard, difficulty: str):
        card.review(difficulty)
        self.reviewed.append(card)


In [15]:
import os

def load_opus_corpus(folder_path):
    en_file = next(f for f in os.listdir(folder_path) if f.endswith('.en'))
    eu_file = next(f for f in os.listdir(folder_path) if f.endswith('.eu'))
    
    with open(os.path.join(folder_path, en_file), 'r', encoding='utf-8') as en_f:
        en_lines = en_f.read().splitlines()
    with open(os.path.join(folder_path, eu_file), 'r', encoding='utf-8') as eu_f:
        eu_lines = eu_f.read().splitlines()
    
    return list(zip(en_lines, eu_lines))

pairs = load_opus_corpus("en-eu.txt/")
print(pairs[:5])


[('I love you. ', 'Maite zaitut. '), ('Congratulations! ', 'Zorionak! '), ("I don't speak Japanese. ", 'Ez dut japoniera hitz egiten. '), ('Thank you very much! ', 'Eskerrik asko. '), ('Thank you very much! ', 'Esker mila! ')]


In [ ]:
if __name__ == "__main__":

    deck = Deck("OPUS Sample", [Flashcard(en, eu) for en, eu in pairs])
    session = ReviewSession(deck)

    while session.has_next():
        card = session.next_card()
        print(f"EN: {card.en.strip()}")
        input("Press Enter to reveal...")
        print(f"EUS: {card.eu.strip()}")
        difficulty = input("How was it? (easy/medium/hard): ").strip().lower()
        session.review_card(card, difficulty)

    print("\nSession complete! Reviewed:")
    for c in session.reviewed:
        print(f"{c.en.strip()} → {c.eu.strip()} | next review in {c.interval:.2f} days")



EN: I love you.
EUS: Maite zaitut.
EN: Congratulations!
EUS: Zorionak!
EN: I don't speak Japanese.
EUS: Ez dut japoniera hitz egiten.
EN: Thank you very much!
EUS: Eskerrik asko.
EN: Thank you very much!
